In [246]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [247]:
raw = pd.read_csv("../data/raw/plays.csv")
raw.columns

Index(['gameId', 'playId', 'playDescription', 'quarter', 'down', 'yardsToGo',
       'possessionTeam', 'defensiveTeam', 'yardlineSide', 'yardlineNumber',
       'gameClock', 'preSnapHomeScore', 'preSnapVisitorScore',
       'playNullifiedByPenalty', 'absoluteYardlineNumber',
       'preSnapHomeTeamWinProbability', 'preSnapVisitorTeamWinProbability',
       'expectedPoints', 'offenseFormation', 'receiverAlignment',
       'playClockAtSnap', 'passResult', 'passLength', 'targetX', 'targetY',
       'playAction', 'dropbackType', 'dropbackDistance', 'passLocationType',
       'timeToThrow', 'timeInTackleBox', 'timeToSack', 'passTippedAtLine',
       'unblockedPressure', 'qbSpike', 'qbKneel', 'qbSneak',
       'rushLocationType', 'penaltyYards', 'prePenaltyYardsGained',
       'yardsGained', 'homeTeamWinProbabilityAdded',
       'visitorTeamWinProbilityAdded', 'expectedPointsAdded', 'isDropback',
       'pff_runConceptPrimary', 'pff_runConceptSecondary', 'pff_runPassOption',
       'pff_pass

Notes:

Numeric columns:
- quarter
- down
- yardsToGo
- urgencyScore - down * yardsToGo
- posessionTeam mapped (possessionTeamPowerRanking)
- defensiveTeam mapped (defensiveTeamPowerRanking)
- gameClock (convert to how many seconds have elapsed)
- preSnapHomeScore
- scoreDifferential between presnaphome and presnapvisitor
- pressureScore - quarter * score differential 
- absoluteyardlinenumber
- timeToThrow
- timeInTackleBox
- passLength

Kmeans categorical columns
- offensiveFormation
- playaction
- dropback type
- passlocationtype
- rush location type
- pff_runConceptPrimary
- pff_runConceptSecondary
- is dropback
- pff_passCoverage
- pff_manZone

In [248]:
print(raw["possessionTeam"].unique())

['CIN' 'HOU' 'KC' 'BAL' 'DET' 'IND' 'ARI' 'PHI' 'DAL' 'GB' 'ATL' 'LV'
 'WAS' 'TEN' 'NE' 'LAC' 'SEA' 'CLE' 'JAX' 'TB' 'LA' 'MIA' 'CHI' 'NO' 'CAR'
 'BUF' 'SF' 'MIN' 'PIT' 'NYJ' 'NYG' 'DEN']


In [249]:
# pft power rankings I found online

team_power_rankings = {
    "SF": 1,
    "BUF": 2,
    "KC": 3,
    "PHI": 4,
    "CIN": 5,
    "MIN": 6,
    "DET": 7,
    "JAX": 8,
    "DAL": 9,
    "LAC": 10,
    "PIT": 11,
    "NYG": 12,
    "BAL": 13,
    "TB": 14,
    "SEA": 15,
    "MIA": 16,
    "GB": 17,
    "NE": 18,
    "CLE": 19,
    "WAS": 20,
    "TEN": 21,
    "CAR": 22,
    "NO": 23,
    "NYJ": 24,
    "ATL": 25,
    "LV": 26,
    "LA": 27,
    "DEN": 28,
    "ARI": 29,
    "CHI": 30,
    "HOU": 31,
    "IND": 32,
}

In [250]:
df = raw[[
    "quarter", 
    "down", 
    "yardsToGo", 
    "preSnapHomeScore", 
    "preSnapVisitorScore", 
    "absoluteYardlineNumber", 
    "timeToThrow", 
    "timeInTackleBox",
    "passLength"
]].copy()
df["urgencyScore"] = df["down"] * df["yardsToGo"]
df["scoreDifferential"] = df["preSnapHomeScore"] - df["preSnapVisitorScore"]
df["pressureScore"] = df["quarter"] * df["scoreDifferential"]
df["passLength"] = df["passLength"].fillna(0)

df.head()

,quarter,down,yardsToGo,preSnapHomeScore,preSnapVisitorScore,absoluteYardlineNumber,timeToThrow,timeInTackleBox,passLength,urgencyScore,scoreDifferential,pressureScore
0,3,1,10,35,17,31,2.990,2.990,6.0,10,18,54
1,4,1,10,17,17,18,1.836,1.836,4.0,10,0,0
2,4,3,12,3,17,30,2.236,2.236,-4.0,36,-14,-56
3,1,2,10,0,0,33,2.202,2.202,-6.0,20,0,0
4,3,2,8,10,10,37,NaN,NaN,0.0,16,0,0


In [251]:
df["offenseRanking"] = raw["possessionTeam"].map(team_power_rankings)
df["defenseRanking"] = raw["defensiveTeam"].map(team_power_rankings)

In [252]:
print(df.head())

   quarter  down  yardsToGo  preSnapHomeScore  preSnapVisitorScore  \
0        3     1         10                35                   17   
1        4     1         10                17                   17   
2        4     3         12                 3                   17   
3        1     2         10                 0                    0   
4        3     2          8                10                   10   

   absoluteYardlineNumber  timeToThrow  timeInTackleBox  passLength  \
0                      31        2.990            2.990         6.0   
1                      18        1.836            1.836         4.0   
2                      30        2.236            2.236        -4.0   
3                      33        2.202            2.202        -6.0   
4                      37          NaN              NaN         0.0   

   urgencyScore  scoreDifferential  pressureScore  offenseRanking  \
0            10                 18             54               5   
1            1

In [253]:
print(df.isna().sum()[df.isna().sum() > 0])

timeToThrow        7419
timeInTackleBox    7207
dtype: int64


In [254]:
df["timeToThrow"] = df["timeToThrow"].fillna(0)
df["timeInTackleBox"] = df["timeInTackleBox"].fillna(0)
df.head()

,quarter,down,yardsToGo,preSnapHomeScore,preSnapVisitorScore,absoluteYardlineNumber,timeToThrow,timeInTackleBox,passLength,urgencyScore,scoreDifferential,pressureScore,offenseRanking,defenseRanking
0,3,1,10,35,17,31,2.990,2.990,6.0,10,18,54,5,25
1,4,1,10,17,17,18,1.836,1.836,4.0,10,0,0,5,9
2,4,3,12,3,17,30,2.236,2.236,-4.0,36,-14,-56,31,21
3,1,2,10,0,0,33,2.202,2.202,-6.0,20,0,0,3,21
4,3,2,8,10,10,37,0.000,0.000,0.0,16,0,0,13,14


In [255]:
def convert_game_clock(clock_str):
    minutes, seconds = map(int, clock_str.split(":"))
    return minutes * 60 + seconds

df["secondsRemainingInQuarter"] = raw["gameClock"].apply(convert_game_clock)
df["secondsElapsedTotal"] = (df["quarter"] - 1) * 900 + (900 - df["secondsRemainingInQuarter"])

In [256]:
df.head()

,quarter,down,yardsToGo,preSnapHomeScore,preSnapVisitorScore,absoluteYardlineNumber,timeToThrow,timeInTackleBox,passLength,urgencyScore,scoreDifferential,pressureScore,offenseRanking,defenseRanking,secondsRemainingInQuarter,secondsElapsedTotal
0,3,1,10,35,17,31,2.990,2.990,6.0,10,18,54,5,25,114,2586
1,4,1,10,17,17,18,1.836,1.836,4.0,10,0,0,5,9,133,3467
2,4,3,12,3,17,30,2.236,2.236,-4.0,36,-14,-56,31,21,120,3480
3,1,2,10,0,0,33,2.202,2.202,-6.0,20,0,0,3,21,568,332
4,3,2,8,10,10,37,0.000,0.000,0.0,16,0,0,13,14,136,2564


In [257]:
cat_cols = [
    "offenseFormation",
    "playAction",
    "dropbackType",
    "passLocationType",
    "rushLocationType",
    "pff_runConceptPrimary",
    "pff_runConceptSecondary",
    "isDropback",
    "pff_passCoverage",
    "pff_manZone"
]

cat_df = raw[cat_cols].copy().fillna("None")
cat_df.head()

,offenseFormation,playAction,dropbackType,passLocationType,rushLocationType,pff_runConceptPrimary,pff_runConceptSecondary,isDropback,pff_passCoverage,pff_manZone
0,EMPTY,False,TRADITIONAL,INSIDE_BOX,None,None,None,True,Cover-3,Zone
1,EMPTY,False,TRADITIONAL,INSIDE_BOX,None,None,None,True,Quarters,Zone
2,SHOTGUN,False,TRADITIONAL,INSIDE_BOX,None,None,None,True,Quarters,Zone
3,SHOTGUN,False,TRADITIONAL,INSIDE_BOX,None,None,None,True,Quarters,Zone
4,PISTOL,True,DESIGNED_RUN,None,INSIDE_LEFT,MAN,READ OPTION,False,Cover-1,Man


In [258]:
cat_df['playAction'] = cat_df['playAction'].astype(int)
cat_df['isDropback'] = cat_df['isDropback'].astype(int)

cat_df.head()

,offenseFormation,playAction,dropbackType,passLocationType,rushLocationType,pff_runConceptPrimary,pff_runConceptSecondary,isDropback,pff_passCoverage,pff_manZone
0,EMPTY,0,TRADITIONAL,INSIDE_BOX,None,None,None,1,Cover-3,Zone
1,EMPTY,0,TRADITIONAL,INSIDE_BOX,None,None,None,1,Quarters,Zone
2,SHOTGUN,0,TRADITIONAL,INSIDE_BOX,None,None,None,1,Quarters,Zone
3,SHOTGUN,0,TRADITIONAL,INSIDE_BOX,None,None,None,1,Quarters,Zone
4,PISTOL,1,DESIGNED_RUN,None,INSIDE_LEFT,MAN,READ OPTION,0,Cover-1,Man


In [259]:
for col in cat_df.columns:
    if cat_df[col].dtype == 'object':
        le = LabelEncoder()
        cat_df[col] = le.fit_transform(cat_df[col])

In [260]:
kmeans = KMeans(n_clusters=8, random_state=42)
cat_df['playStyleCluster'] = kmeans.fit_predict(cat_df)
cat_df.head()
print(cat_df['playStyleCluster'].value_counts())

playStyleCluster
1    3966
3    3315
2    2903
6    2767
7     949
0     929
4     895
5     400
Name: count, dtype: int64


In [261]:
df["playStyleCluster"] = cat_df["playStyleCluster"].values
df["yardsGained"] = raw["yardsGained"].values

In [262]:
print(df.columns)
df.head(10)

Index(['quarter', 'down', 'yardsToGo', 'preSnapHomeScore',
       'preSnapVisitorScore', 'absoluteYardlineNumber', 'timeToThrow',
       'timeInTackleBox', 'passLength', 'urgencyScore', 'scoreDifferential',
       'pressureScore', 'offenseRanking', 'defenseRanking',
       'secondsRemainingInQuarter', 'secondsElapsedTotal', 'playStyleCluster',
       'yardsGained'],
      dtype='object')


,quarter,down,yardsToGo,preSnapHomeScore,preSnapVisitorScore,absoluteYardlineNumber,timeToThrow,timeInTackleBox,passLength,urgencyScore,scoreDifferential,pressureScore,offenseRanking,defenseRanking,secondsRemainingInQuarter,secondsElapsedTotal,playStyleCluster,yardsGained
0,3,1,10,35,17,31,2.990,2.990,6.0,10,18,54,5,25,114,2586,1,9
1,4,1,10,17,17,18,1.836,1.836,4.0,10,0,0,5,9,133,3467,2,4
2,4,3,12,3,17,30,2.236,2.236,-4.0,36,-14,-56,31,21,120,3480,2,6
3,1,2,10,0,0,33,2.202,2.202,-6.0,20,0,0,3,21,568,332,2,4
4,3,2,8,10,10,37,0.000,0.000,0.0,16,0,0,13,14,136,2564,4,-1
5,3,2,6,15,31,39,0.000,0.000,0.0,12,-16,-48,7,15,855,1845,3,3
6,4,1,10,26,3,50,0.000,0.000,0.0,10,23,92,32,18,29,3571,3,5
7,4,3,12,16,26,82,0.000,0.000,0.0,36,-10,-40,29,22,35,3565,2,-1
8,4,3,12,28,38,45,1.568,1.568,-6.0,36,-10,-40,4,7,771,2829,6,0
9,2,3,8,6,7,45,3.203,3.203,15.0,24,-1,-2,9,20,322,1478,2,15


In [263]:
numeric_cols_to_normalize = [
    "yardsToGo",
    "urgencyScore",
    "absoluteYardlineNumber",
    "scoreDifferential",
    "pressureScore",
    "secondsRemainingInQuarter",
    "secondsElapsedTotal",
    "timeToThrow",
    "timeInTackleBox",
    "preSnapHomeScore",
    "preSnapVisitorScore",
    "offenseRanking",
    "defenseRanking",
    "passLength"
]

scaler = StandardScaler()
df[numeric_cols_to_normalize] = scaler.fit_transform(df[numeric_cols_to_normalize])
df.head()



,quarter,down,yardsToGo,preSnapHomeScore,preSnapVisitorScore,absoluteYardlineNumber,timeToThrow,timeInTackleBox,passLength,urgencyScore,scoreDifferential,pressureScore,offenseRanking,defenseRanking,secondsRemainingInQuarter,secondsElapsedTotal,playStyleCluster,yardsGained
0,3,1,0.393677,2.507782,0.753479,-1.209175,0.969961,1.062836,0.224264,-0.421079,1.808632,1.666333,-1.258960,0.907228,-1.162002,0.656919,1,9
1,4,1,0.393677,0.606681,0.753479,-1.743599,0.227520,0.263292,-0.015304,-0.421079,-0.129905,-0.117837,-1.258960,-0.847351,-1.092144,1.482502,2,4
2,4,3,0.905377,-0.871954,0.753479,-1.250285,0.484865,0.540431,-0.973572,2.270531,-1.637656,-1.968089,1.564528,0.468583,-1.139942,1.494684,2,6
3,1,2,0.393677,-1.188804,-1.089461,-1.126956,0.462991,0.516874,-1.213139,0.614155,-0.129905,-0.117837,-1.476151,0.468583,0.507240,-1.455301,2,4
4,3,2,-0.118024,-0.132637,-0.005379,-0.962518,-0.953696,-1.008771,-0.494438,0.200061,-0.129905,-0.117837,-0.390194,-0.299045,-1.081114,0.636303,4,-1


In [264]:
one_hot = pd.get_dummies(df["playStyleCluster"], prefix="playStyleCluster")
df = pd.concat([df.drop(columns=["playStyleCluster"]), one_hot], axis=1)
df = df.astype(float)
df.head()

,quarter,down,yardsToGo,preSnapHomeScore,preSnapVisitorScore,absoluteYardlineNumber,timeToThrow,timeInTackleBox,passLength,urgencyScore,...,secondsElapsedTotal,yardsGained,playStyleCluster_0,playStyleCluster_1,playStyleCluster_2,playStyleCluster_3,playStyleCluster_4,playStyleCluster_5,playStyleCluster_6,playStyleCluster_7
0,3.0,1.0,0.393677,2.507782,0.753479,-1.209175,0.969961,1.062836,0.224264,-0.421079,...,0.656919,9.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
1,4.0,1.0,0.393677,0.606681,0.753479,-1.743599,0.227520,0.263292,-0.015304,-0.421079,...,1.482502,4.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,4.0,3.0,0.905377,-0.871954,0.753479,-1.250285,0.484865,0.540431,-0.973572,2.270531,...,1.494684,6.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
3,1.0,2.0,0.393677,-1.188804,-1.089461,-1.126956,0.462991,0.516874,-1.213139,0.614155,...,-1.455301,4.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4,3.0,2.0,-0.118024,-0.132637,-0.005379,-0.962518,-0.953696,-1.008771,-0.494438,0.200061,...,0.636303,-1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


In [265]:
df.to_csv("../data/processed/plays_processed_kmeans.csv", index=False)